# Hybrid retrieval sanity checks

Run from a fresh Python kernel after `uv sync`. This notebook uses local BGE + SQLite, saves inspection artifacts, and makes no answer-generation API calls. Hybrid is the default. The switches independently disable each path. Results establish pipeline behavior, not clinical accuracy.

In [1]:
import json
import sys
import time
from pathlib import Path
from dataclasses import asdict
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
from mobile_rag.retrieval import new_run_dir, write_json
from mobile_rag.retrieval_hybrid import HybridRetriever, RetrievalConfig, build_dense, latest_index
from mobile_rag.context_preparation import prepare_context
from mobile_rag.answer_generation import generate_answer

ENABLE_BM25 = True
ENABLE_EMBEDDINGS = True
BUILD_INDEX = False  # True explicitly downloads the pinned model and builds a new vector bundle.
INDEX_DIR = None  # Set an explicit compatible bundle to inspect an older run.
RUN_BENCHMARK = True
OUTPUT_ROOT = ROOT / 'artifacts/03_2_hybrid_sanity'
config = RetrievalConfig(ENABLE_BM25, ENABLE_EMBEDDINGS)
config.validate()
print(asdict(config))

{'enable_bm25': True, 'enable_embeddings': True, 'candidate_limit': 20}


## Resolve or build the index

The default reuses a completed dense bundle. With BUILD_INDEX=True, the lexical source is copied into a new bundle and all source chunks are embedded. Existing bundles remain intact. Ordinary searches require cached model files and do not download them.

In [2]:
if BUILD_INDEX:
    source = latest_index(ROOT, RetrievalConfig(True, False))
    INDEX_DIR = build_dense(source, ROOT / 'artifacts/03_retrieval_enhanced')
elif INDEX_DIR is None:
    INDEX_DIR = latest_index(ROOT, config)
INDEX_DIR = Path(INDEX_DIR)
RUN_DIR = new_run_dir(OUTPUT_ROOT)
manifest = json.loads((INDEX_DIR / 'dense_manifest.json').read_text()) if (INDEX_DIR / 'dense_manifest.json').exists() else None
display({'index': str(INDEX_DIR), 'output': str(RUN_DIR), 'dense_manifest': manifest})

{'index': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\03_retrieval_enhanced\\20260913_135906',
 'output': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\03_2_hybrid_sanity\\20260913_141554',
 'dense_manifest': {'bundle_identity': 'e4b7bbef61e925b6f63865ba7d3efd7d56c395ac300ad019904b08369c1cb02c',
  'encoder': {'dimensions': 384,
   'model': 'BAAI/bge-small-en-v1.5',
   'model_sha256': '828e1496d7fabb79cfa4dcd84fa38625c0d3d21da474a00f08db0f559940cf35',
   'normalization': 'l2',
   'pooling': 'cls',
   'query_prefix': 'Represent this sentence for searching relevant passages: ',
   'revision': '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a',
   'tokenizer_sha256': 'd241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66'},
  'index_identity': '1e502c96a8e275f7cedb09e595e529214dba7d6fe9c274b854b35ec250f21f11',
  'retrieval_sha256': '727b37772d630e47b4b61eb47eeacef36d33716a3b61625f23d49ac5b4c9a1fa',
  'schema': 'bge-dense/v1',
  'stride_tokens': 384,
  'table_li

## Inspect the selected mode

Check the source excerpts yourself. Dense similarity and RRF are ranking signals, not evidence that a clinical answer is correct.

In [3]:
QUESTION = 'When should CPAP be avoided?'
with HybridRetriever(INDEX_DIR, config) as retriever:
    result = retriever.search(QUESTION)
    context = prepare_context(retriever, result, retriever.expand(result))
    assert context['status'] in ('ready', 'budget_blocked', 'empty')
write_json(RUN_DIR / 'selected_retrieval.json', result)
write_json(RUN_DIR / 'selected_context.json', context)
for hit in result['hits']:
    display({'rank': hit['rank'], 'paths': hit['branch_ranks'], 'dense_score': hit['dense_score'],
             'source': hit['document']['markdown_path'], 'citation': hit['citation'],
             'text': hit['chunk']['retrieval_text'][:1800]})
print('Context status:', context['status'])

C:\Users\PK\Desktop\projects\mobile_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'rank': 1,
 'paths': {'bm25': 3, 'embeddings': 1},
 'dense_score': 0.7842189073562622,
 'source': 'data/md_docs/Bubble-CPAP-guidelines-2017.md',
 'citation': {'chunk_id': 'chunk_0887f5a6fe54ca4a73ae',
  'citation_target_id': 'citation_428ee2d1d353ba00352b',
  'declared_pages': [7],
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'pdf_coordinate_status': 'unavailable',
  'source_passage_ids': ['passage_05d56a38e308aab0b0ad',
   'passage_330e0d088f0fe93facb8',
   'passage_a2548c624401bee026ab']},
 'text': '# Guidelines for Use of Bubble-CPAP Concentrators\r\n\n\n### Trials off CPAP and when to stop CPAP\r\n\n\nWhen children who are clinically stable (low respiratory distress score and SpO₂ >92%), CPAP should be disconnected for 10–15 minutes, the child should be put on standard-flow oxygen, and clinical signs and SpO₂ should be carefully examined to assess whether CPAP is still required.\r\n\n\nTrials off CPAP are best done first thing in the m

{'rank': 2,
 'paths': {'bm25': 2, 'embeddings': 2},
 'dense_score': 0.7698951959609985,
 'source': 'data/md_docs/Bubble-CPAP-guidelines-2017.md',
 'citation': {'chunk_id': 'chunk_5fafebb4b9edb42b6847',
  'citation_target_id': 'citation_c4c8f0e679e8590bcd30',
  'declared_pages': [7],
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'pdf_coordinate_status': 'unavailable',
  'source_passage_ids': ['passage_a88d985ac5034dff1baf']},
 'text': '# Guidelines for Use of Bubble-CPAP Concentrators\r\n\n\n### Trials off CPAP and when to stop CPAP\r\n\n\nSome children will become hypoxaemic rapidly when taken off CPAP; this is a marker of very severe disease. Increase their oxygen or put them back on CPAP immediately. You should be at the bedside, monitor SpO₂, and watch the child for cyanosis or severe respiratory distress. Instruct parents and nursing staff on what to observe.\r\n'}

{'rank': 3,
 'paths': {'bm25': 5, 'embeddings': 4},
 'dense_score': 0.7616294026374817,
 'source': 'data/md_docs/Bubble-CPAP-guidelines-2017.md',
 'citation': {'chunk_id': 'chunk_18d60ca3aefb998a66b9',
  'citation_target_id': 'citation_7c2b0aa29f719f79d1e8',
  'declared_pages': [8],
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'pdf_coordinate_status': 'unavailable',
  'source_passage_ids': ['passage_f855eb3935ee23dad0f0',
   'passage_f1f5e9e695f7a4059da4']},
 'text': '# Guidelines for Use of Bubble-CPAP Concentrators\r\n\n\n### Steps in CPAP weaning\r\n\n\n**Children with acute respiratory disease should not be discharged until:**\r\n\n\n1. SpO₂ has been stable at >90% while breathing room air for at least 24 hours.\r\n2. Danger signs have resolved.\r\n3. Appropriate home treatment can be organized.\r\n4. The parents or guardian understand the danger signs to look for, when to return if the child becomes sicker, and when to return for a pla

{'rank': 4,
 'paths': {'bm25': 6, 'embeddings': 6},
 'dense_score': 0.7483729720115662,
 'source': 'data/md_docs/Bubble-CPAP-guidelines-2017.md',
 'citation': {'chunk_id': 'chunk_1e72f93fc30a1f2296d2',
  'citation_target_id': 'citation_6c1451a3b604e3a29dce',
  'declared_pages': [8],
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'pdf_coordinate_status': 'unavailable',
  'source_passage_ids': ['passage_fd419414f024dca0639f']},
 'text': '# Guidelines for Use of Bubble-CPAP Concentrators\r\n\n\n### Steps in CPAP weaning\r\n\n\nIf the child on CPAP is clinically stable (low respiratory distress score and SpO₂ >92%), disconnect from CPAP as described below for 10–15 minutes and carefully examine for changes in clinical signs and SpO₂ to assess whether CPAP is still required:\r\n'}

{'rank': 5,
 'paths': {'bm25': 10, 'embeddings': 3},
 'dense_score': 0.7657065987586975,
 'source': 'data/md_docs/Bubble-CPAP-guidelines-2017.md',
 'citation': {'chunk_id': 'chunk_8524539cc97f7618746c',
  'citation_target_id': 'citation_2ea177b80e9d8d2b7d1b',
  'declared_pages': [2],
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'pdf_coordinate_status': 'unavailable',
  'source_passage_ids': ['passage_fd8f8a9b449b8c997803',
   'passage_7e4914dbeb814e635ac8',
   'passage_20fda77cd3cdeec283ad']},
 'text': '# Guidelines for Use of Bubble-CPAP Concentrators\r\n\n\n### Conditions for which CPAP is not usually useful or is contraindicated\r\n\n\nCPAP is contraindicated if a child has pneumothorax, as it can make the air leak worse. In some conditions, such as staphylococcal pneumonia with pneumatocoeles on chest X-ray, CPAP can increase the risk of pneumothorax, so caution is needed.\r\n\n\nCPAP will not be effective in an unconscious child who is

Context status: ready


## Compare the independent switches

This comparison intentionally tests all three modes when a dense bundle exists. The selected-mode cell above follows only your switches. An embeddings-only result is not restricted to the lexical candidate list.

In [4]:
modes = {'bm25_only': RetrievalConfig(True, False)}
if manifest:
    modes.update(embeddings_only=RetrievalConfig(False, True), hybrid=RetrievalConfig(True, True))
queries = [QUESTION, 'What is the oxygen saturation threshold for a critically ill child?',
           'When should iron be started in severe acute malnutrition?', '5 kg F-75 feeding volume']
comparisons = []
for name, option in modes.items():
    with HybridRetriever(INDEX_DIR, option) as retriever:
        for question in queries:
            found = retriever.search(question)
            expected = ({'bm25'} if option.enable_bm25 else set()) | ({'embeddings'} if option.enable_embeddings else set())
            assert set(found['branches']) == expected
            assert len({h['chunk']['chunk_id'] for h in found['hits']}) == len(found['hits'])
            comparisons.append({'mode': name, 'question': question, 'seconds': found['search_seconds'],
                                'chunk_ids': [h['chunk']['chunk_id'] for h in found['hits']],
                                'paths': [h['branch_ranks'] for h in found['hits']]})
display(comparisons)
write_json(RUN_DIR / 'mode_comparison.json', comparisons)
try:
    RetrievalConfig(False, False).validate()
except ValueError as error:
    print('Both-off correctly rejected:', error)
else:
    raise AssertionError('Both-off was accepted')

[{'mode': 'bm25_only',
  'question': 'When should CPAP be avoided?',
  'seconds': 0.12490410008467734,
  'chunk_ids': ['chunk_1416ee61798ba3a3e4d3',
   'chunk_5fafebb4b9edb42b6847',
   'chunk_0887f5a6fe54ca4a73ae',
   'chunk_cdfc888b6426f9dd226c',
   'chunk_18d60ca3aefb998a66b9'],
  'paths': [{'bm25': 1}, {'bm25': 2}, {'bm25': 3}, {'bm25': 4}, {'bm25': 5}]},
 {'mode': 'bm25_only',
  'question': 'What is the oxygen saturation threshold for a critically ill child?',
  'seconds': 0.13088369998149574,
  'chunk_ids': ['chunk_58e93f4c1e0561652332',
   'chunk_5fc8f520c5207c7e1c62',
   'chunk_7ef09b402f1dc0e2890c',
   'chunk_6e5130659985493b2e69',
   'chunk_3b54d3d15a5ab60d7ddb'],
  'paths': [{'bm25': 1}, {'bm25': 2}, {'bm25': 3}, {'bm25': 4}, {'bm25': 5}]},
 {'mode': 'bm25_only',
  'question': 'When should iron be started in severe acute malnutrition?',
  'seconds': 0.13108150009065866,
  'chunk_ids': ['chunk_d8bdeb4c287978782dac',
   'chunk_c2d7292c063bd048c9ee',
   'chunk_9f8ef89a03b37ac7f0

Both-off correctly rejected: Enable at least one retrieval path: BM25 or embeddings


## Verify empty lexical results do not block dense retrieval

The artificial query below has no expected medical meaning. This is only a routing check: dense search still returns nearest neighbors. It also demonstrates why nearest neighbors alone cannot establish answerability.

In [5]:
if manifest:
    with HybridRetriever(INDEX_DIR) as retriever:
        independent = retriever.search('unfindablewordxyz')
        assert independent['branches']['bm25'] == []
        assert independent['branches']['embeddings']
        assert independent['hits']
    print('Dense path returned candidates despite zero lexical matches.')
else:
    print('Dense routing check skipped: lexical-only bundle.')
dry = generate_answer(context, live=False)
assert not dry['live_request_sent']
print('Generation contract dry run:', dry['status'])

Dense path returned candidates despite zero lexical matches.
Generation contract dry run: dry_run


## Optional Q_S1 retrieval comparison

Uses questions only, never reference answers for retrieval. Records candidate changes and timing; no automatic accuracy claim is made. First-use timings include normal warm-up effects.

In [6]:
benchmark_rows = []
if RUN_BENCHMARK:
    questions = json.loads((ROOT / 'data/questions/Q_S1.json').read_text(encoding='utf-8-sig'))['questions']
    for name, option in modes.items():
        with HybridRetriever(INDEX_DIR, option) as retriever:
            for row in questions:
                found = retriever.search(row['question'])
                benchmark_rows.append({'id': row['id'], 'mode': name, 'status': found['status'],
                                       'seconds': found.get('search_seconds'),
                                       'chunk_ids': [h['chunk']['chunk_id'] for h in found['hits']]})
    write_json(RUN_DIR / 'benchmark_retrieval.json', benchmark_rows)
summary = {'index': str(INDEX_DIR), 'selected_config': asdict(config),
           'modes_checked': list(modes), 'benchmark_searches': len(benchmark_rows),
           'checks_passed': True, 'clinical_accuracy': 'not_measured',
           'answer_generation_api_calls': 0}
write_json(RUN_DIR / 'summary.json', summary)
display(summary)
print('Saved artifacts:', RUN_DIR)

{'index': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\03_retrieval_enhanced\\20260913_135906',
 'selected_config': {'enable_bm25': True,
  'enable_embeddings': True,
  'candidate_limit': 20},
 'modes_checked': ['bm25_only', 'embeddings_only', 'hybrid'],
 'benchmark_searches': 300,
 'checks_passed': True,
 'clinical_accuracy': 'not_measured',
 'answer_generation_api_calls': 0}

Saved artifacts: C:\Users\PK\Desktop\projects\mobile_rag\artifacts\03_2_hybrid_sanity\20260913_141554
